# Research notebook

Read the module README before executing. Textual outputs below are saved historical snapshots; the report follows the original project document and was not recalculated during publication cleanup. Setup paths have been made portable. Embedded media and machine-specific diagnostic output are omitted from this public-facing copy.


In [ ]:
# Portable project paths; this cell does not load a model.
from pathlib import Path
import sys

_candidates = (Path.cwd(), *Path.cwd().parents)
_project_root = next((p for p in _candidates if (p / "project_paths.py").is_file()), None)
if _project_root is None:
    raise RuntimeError("Start Jupyter from the retrieval repository or one of its subdirectories.")
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from project_paths import (GALLERY_DIR, SIGLIP_MODEL, REFERENCE_CLASS_FOLDERS,
                           reference_class_indices)
# Experiment settings. Edit REFERENCE_CLASS_FOLDERS in project_paths.py for your data.
target_classes = list(REFERENCE_CLASS_FOLDERS)
CLIP_MODEL = "ViT-B/32"
REFERENCE_COUNT = 20



In [1]:
import os
from PIL import Image
import torch
import clip
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load(CLIP_MODEL, device=device)
import numpy as np
from pathlib import Path

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import random
from transformers import BertTokenizer, BertForSequenceClassification, CLIPProcessor, CLIPModel
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel, BitsAndBytesConfig,AutoTokenizer
import accelerate
import sentencepiece

In [2]:
#区分数据集为参考图和图片池
def split(dataset, target_label, ref_number, seed=0):
    #获取所有图片索引，在目标类别选取参考图，再分开数据集
    all_indices=list(range(len(dataset)))
    target_indices=[idx for idx in all_indices if dataset[idx][1]==target_label]
    random.seed(seed)
    ref_indices=random.sample(target_indices, ref_number)
    gallery_indices=[idx for idx in all_indices if idx not in ref_indices]
    return Subset(dataset,ref_indices), Subset(dataset,gallery_indices)
def f1_score(similarity,gallery_label,threshold,target_label):
    #pos是布尔值，在simliarity中使用可以获得目标标签的变量
    pos_bool=(gallery_label==target_label)
    neg_bool=(gallery_label!=target_label)
    pos=similarity[pos_bool]
    neg=similarity[neg_bool]
    TP=sum(pos>=threshold)
    FP=sum(neg>=threshold)
    FN=sum(pos<threshold)
    precision=TP/(TP+FP+0.0001)
    recall=TP/(TP+FN+0.0001)
    f1=2*precision*recall/(precision+recall+0.0001)
    return f1,precision,recall
def find_threshold(similarity,gallery_label,target_label):
    thresholds=np.linspace(np.min(similarity),np.max(similarity),100)
    best_f1=0
    best_precision=0
    best_recall=0
    for threshold in thresholds:
        f1,precision,recall=f1_score(similarity,gallery_label,threshold,target_label)
        if f1 > best_f1:
            best_f1=f1
            best_precision=precision
            best_recall=recall
    return best_f1,best_precision,best_recall


In [4]:
model = AutoModel.from_pretrained(SIGLIP_MODEL).to(device).eval()
processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
tokenizer=AutoTokenizer.from_pretrained(SIGLIP_MODEL)
inputs = tokenizer([f"a photo of a {target_classes[0].lower()}"], padding="max_length", return_tensors="pt").to(device)
with torch.no_grad():
    text_features = model.get_text_features(**inputs)
print(text_features)

tensor([[ 2.3365e-01, -2.2048e-01,  2.1819e-01, -7.9777e-01, -3.3047e-01,
          4.1288e-01, -2.5245e-01, -2.1833e-01, -4.5122e-01, -5.2966e-01,
         -6.0505e-01, -3.7748e-01,  6.4617e-01, -3.2685e-01,  7.8214e-01,
         -2.9809e-01, -1.2639e-01,  3.0222e-01,  5.1676e-01, -2.5068e-01,
          8.7591e-02, -2.8219e-01, -4.5590e-01,  2.0945e-01,  2.9299e-01,
         -8.9178e-02, -9.4117e-02, -3.4830e-01,  4.9425e-01,  2.4337e-01,
         -2.1808e-01, -2.5519e-01,  3.6845e-01,  1.6177e-01, -1.2874e-01,
         -1.1997e-01, -6.6883e-01,  8.3592e-01,  3.2674e-01,  3.3155e-01,
         -2.9305e-01,  2.7349e-01,  1.8591e-01,  4.2283e-02,  4.0402e-01,
          7.5408e-01,  1.0318e-01, -2.6101e-01,  2.2917e-01, -5.8063e-01,
          4.9086e-01, -3.9626e-01,  2.9467e-02, -7.4409e-01,  4.0074e-01,
         -8.2527e-01,  3.7708e-01,  1.1817e-01, -5.9086e-01,  5.4289e-02,
         -4.3012e-01, -2.3149e-01,  5.6052e-01,  2.7073e-01,  1.1728e-01,
          2.3265e-01,  5.0444e-01,  2.

In [8]:
def main():
 model = AutoModel.from_pretrained(SIGLIP_MODEL).to(device).eval()
 processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
 tokenizer=AutoTokenizer.from_pretrained(SIGLIP_MODEL)

 print('模型加载完成')
 def clip_transform(image):
    # 1. 用processor处理图像，得到字典
        inputs = processor(images=image, return_tensors="pt")
    # 2. 提取pixel_values并移除batch维度（从[1,3,224,224]转为[3,224,224]）
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values
 dataset=ImageFolder(root=str(GALLERY_DIR),transform=clip_transform)
 print('数据集加载完成')
 ref_number=REFERENCE_COUNT
 for target_class in target_classes:
        print(target_class)
        dic = reference_class_indices(dataset.class_to_idx)
        #target_class='Dog'
        #ref_number=10
        target_label=dic[target_class]
        text={f'a photo of a {target_class}'}
        #使用imageFolder导入数据集
        #提取文本特征
        inputs = tokenizer([f"a photo of a {target_class}"], padding="max_length", return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        ref_dataset,gallery_dataset=split(dataset,target_label,ref_number)
        ref = torch.utils.data.DataLoader(ref_dataset,shuffle=False)
        gallery=torch.utils.data.DataLoader(gallery_dataset, batch_size=256, num_workers=0, shuffle=False)
        print('数据集创建完成')
        model.eval()
        gallery_features=[]
        ref_features=[]
        with torch.no_grad():
            for pixel_values, _ in gallery:  # 直接获取pixel_values（已由clip_transform处理）
                pixel_values = pixel_values.to(device)
                batch_features = model.get_image_features(pixel_values=pixel_values)  # 正确参数
                batch_features = batch_features / batch_features.norm(dim=1, keepdim=True)
                gallery_features.append(batch_features)
            print('特征创建完成')
            for pixel_values, _ in ref:
                pixel_values = pixel_values.to(device)
                batch_features = model.get_image_features(pixel_values=pixel_values)  # 正确参数
                batch_features = batch_features / batch_features.norm(dim=1, keepdim=True)
                ref_features.append(batch_features)
            print('特征创建完成')
    # 合并所有批次的特征
        gallery_features = torch.cat(gallery_features, dim=0)
        ref_features = torch.cat(ref_features, dim=0)
        ref_features=ref_features.mean(dim=0, keepdim=True)
        ref_features=(ref_features+text_features)/2
    # 计算相似度矩阵
        similarity = gallery_features @ ref_features.T
        similarity=  similarity.cpu().numpy()
        gallery_label = [sample[1] for sample in gallery_dataset]  # sample[1]是标签
        gallery_label = np.array(gallery_label)
        best_f1,best_precision,best_recall=find_threshold(similarity,gallery_label,target_label)
        print(f'{target_class},{ref_number}:f1:{best_f1},precision:{best_precision},recall:{best_recall}')

In [9]:
main()

模型加载完成
数据集加载完成
Dog
数据集创建完成
特征创建完成
特征创建完成
Dog,20:f1:[0.88353184],precision:[0.95483809],recall:[0.82222177]
Duck
数据集创建完成
特征创建完成
特征创建完成
Duck,20:f1:[0.84205532],precision:[0.76712294],recall:[0.93333281]
Erhu
数据集创建完成
特征创建完成
特征创建完成
Erhu,20:f1:[0.92392621],precision:[0.97530804],recall:[0.87777729]
Piano
数据集创建完成
特征创建完成
特征创建完成
Piano,20:f1:[0.82886614],precision:[0.73191458],recall:[0.95555502]
Porcelain
数据集创建完成
特征创建完成
特征创建完成
Porcelain,20:f1:[0.82474745],precision:[0.8010467],recall:[0.84999953]
